In [1]:
#imports
import pandas as pd

In [2]:
#CSV - loan.csv
df_csv = pd.read_csv("/home/desktop/Optimizing-P2P-Loan-Thresholds-for-Borrowers-Using-LendingClub-Data-/dataset/loan.csv", low_memory=False)

#XLSX - LCDataDictionary
df_xlsx = pd.read_excel("/home/desktop/Optimizing-P2P-Loan-Thresholds-for-Borrowers-Using-LendingClub-Data-/dataset/LCDataDictionary.xlsx")

In [3]:
#Printe a boere of the data
print("Before:", df_csv.shape)

Before: (2260668, 145)


In [4]:
#show the lodaed data:
#df_csv.shape
df_csv.head()
#df_csv.columns.tolist()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,NaN,NaN,2500,2500,2500.0,36 months,13.56,84.92,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,30000,30000,30000.0,60 months,18.94,777.23,D,D2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,5000,5000,5000.0,36 months,17.97,180.69,D,D1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,4000,4000,4000.0,36 months,18.94,146.51,D,D2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,30000,30000,30000.0,60 months,16.14,731.78,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
#Data Cleaning 
drop_cols = ['member_id', 'id','emp_title','url','desc','title','zip_code','policy_code',
 'funded_amnt_inv','out_prncp_inv','total_pymnt_inv',
 'last_pymnt_d','last_pymnt_amnt','next_pymnt_d','last_credit_pull_d',
 'annual_inc_joint','dti_joint','verification_status_joint','revol_bal_joint',
 'sec_app_earliest_cr_line','sec_app_inq_last_6mths','sec_app_mort_acc',
 'sec_app_open_acc','sec_app_revol_util','sec_app_open_act_il',
 'sec_app_num_rev_accts','sec_app_chargeoff_within_12_mths',
 'sec_app_collections_12_mths_ex_med','sec_app_mths_since_last_major_derog',
 'hardship_flag','hardship_type','hardship_reason','hardship_status','deferral_term',
 'hardship_amount','hardship_start_date','hardship_end_date','payment_plan_start_date',
 'hardship_length','hardship_dpd','hardship_loan_status',
 'orig_projected_additional_accrued_interest','hardship_payoff_balance_amount',
 'hardship_last_payment_amount','debt_settlement_flag','debt_settlement_flag_date',
 'settlement_status','settlement_date','settlement_amount','settlement_percentage',
 'settlement_term', 'installment', 
 'out_prncp', 'total_rec_prncp', 'total_rec_int', 
 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee']

clean_df = df_csv.drop(columns=drop_cols).copy()

# Only keep loans that reached a final state. This keeps our model training on loans that have completed or defaulted. 
final_status = ['Fully Paid', 'Charged Off', 'Default',
                'Late (31-120 days)', 'Late (16-30 days)', 'In Grace Period']
clean_df = clean_df[clean_df['loan_status'].isin(final_status)].copy()

#drop loan status. a paid loan will be profitable usually
clean_df = clean_df.drop(columns='loan_status').copy()

clean_df.shape

(1338224, 86)

In [6]:
#Now our feature we want. Is this loan profitable? profitable = total_payment > funded_amnt
clean_df['profitable'] = (clean_df['total_pymnt'] > clean_df['funded_amnt']).astype(int)
# Check the balance of your target
clean_df['profitable'].value_counts()

profitable
1    1063637
0     274587
Name: count, dtype: int64

In [11]:
#Dropping leaky column. 
leaky_columns = ['total_pymnt']
clean_df = clean_df.drop(columns=leaky_columns).copy()

In [12]:
clean_df.head()

,loan_amnt,funded_amnt,term,int_rate,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,...,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,disbursement_method,profitable
100,30000,30000,36 months,22.35,D,D5,5 years,MORTGAGE,100000.0,Source Verified,...,89.5,33.3,1.0,0.0,527120.0,98453.0,28600.0,101984.0,Cash,1
152,40000,40000,60 months,16.14,C,C4,< 1 year,MORTGAGE,45000.0,Verified,...,100.0,42.9,0.0,0.0,344802.0,161720.0,45700.0,167965.0,Cash,1
170,20000,20000,36 months,7.56,A,A3,10+ years,MORTGAGE,100000.0,Not Verified,...,94.7,20.0,0.0,0.0,622183.0,71569.0,85100.0,74833.0,Cash,1
186,4500,4500,36 months,11.31,B,B3,10+ years,RENT,38500.0,Not Verified,...,91.7,0.0,0.0,0.0,53795.0,29137.0,15100.0,24595.0,Cash,1
215,8425,8425,36 months,27.27,E,E5,3 years,MORTGAGE,450000.0,Verified,...,100.0,50.0,0.0,0.0,768304.0,189194.0,45800.0,189054.0,Cash,1


In [13]:
#Show non numerical vlaues and what they look like. 
non_numeric = [c for c in clean_df.columns if not pd.api.types.is_numeric_dtype(clean_df[c])]
for c in non_numeric:
    print(c, clean_df[c].dtype)

for c in non_numeric:
    print("=== ", c, " ===")
    print(clean_df[c].value_counts(dropna=False).head(10))
    print()

term object
grade object
sub_grade object
emp_length object
home_ownership object
verification_status object
issue_d object
pymnt_plan object
purpose object
addr_state object
earliest_cr_line object
initial_list_status object
application_type object
disbursement_method object
===  term  ===
term
36 months    1008975
60 months     329249
Name: count, dtype: int64

===  grade  ===
grade
B    387748
C    382098
A    228494
D    203067
E     94929
F     32547
G      9341
Name: count, dtype: int64

===  sub_grade  ===
sub_grade
C1    85065
B4    82335
B5    81672
B3    80788
C2    79085
C3    75079
C4    74801
B2    72922
B1    70031
C5    68068
Name: count, dtype: int64

===  emp_length  ===
emp_length
10+ years    439321
2 years      120993
< 1 year     107722
3 years      107064
1 year        88083
5 years       83760
4 years       80155
NaN           78394
6 years       62448
8 years       60394
Name: count, dtype: int64

===  home_ownership  ===
home_ownership
MORTGAGE    660800
RENT  

In [14]:
clean_df.shape

(1338224, 86)

In [15]:
#Remove the term months into just an integer of months
clean_df['term_months'] = clean_df['term'].str.extract(r'(\d+)').astype(int)
clean_df = clean_df.drop(columns=['term'])

In [16]:
#Grade and subgrade get mapped to integer values, with A = 0 (best credit) G = 6 (worst credit)
grade_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6}
clean_df['grade'] = clean_df['grade'].map(grade_map)

sub_grade_map = {
    'A1':0,'A2':1,'A3':2,'A4':3,'A5':4,
    'B1':5,'B2':6,'B3':7,'B4':8,'B5':9,
    'C1':10,'C2':11,'C3':12,'C4':13,'C5':14,
    'D1':15,'D2':16,'D3':17,'D4':18,'D5':19,
    'E1':20,'E2':21,'E3':22,'E4':23,'E5':24,
    'F1':25,'F2':26,'F3':27,'F4':28,'F5':29,
    'G1':30,'G2':31,'G3':32,'G4':33,'G5':34
}
clean_df['sub_grade'] = clean_df['sub_grade'].map(sub_grade_map)

In [17]:
#cleaning up employment length
emp_map = {
    '< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3, '4 years': 4,
    '5 years': 5, '6 years': 6, '7 years': 7, '8 years': 8, '9 years': 9,
    '10+ years': 10
}
clean_df['emp_length'] = clean_df['emp_length'].map(emp_map)

In [18]:
#cleaning up home_ownership,  purpose, addr_state, verification_status 
#one-hot was used to give a binary values to indacte the value of the feature. 
#numerical values are not used so that the model deos attempt to asscoaite. OWN > MORTGAGE > RENT. 3 > 2 > 1

# Binary columns -> simple 0/1
clean_df['pymnt_plan'] = (clean_df['pymnt_plan'] == 'y').astype(int)
clean_df['initial_list_status'] = (clean_df['initial_list_status'] == 'f').astype(int)
clean_df['application_type'] = (clean_df['application_type'] == 'Joint App').astype(int)
clean_df['disbursement_method'] = (clean_df['disbursement_method'] == 'DirectPay').astype(int)

# Multi-category nominal columns -> one-hot
clean_df = pd.get_dummies(clean_df,
                          columns=['home_ownership','purpose','addr_state','verification_status'],
                          drop_first=True, dtype=int)

In [19]:
# --- Date handling: issue_d and earliest_cr_line ---

# Parse the Mon-YYYY strings into proper datetime objects
clean_df['issue_d'] = pd.to_datetime(clean_df['issue_d'], format='%b-%Y')
clean_df['earliest_cr_line'] = pd.to_datetime(clean_df['earliest_cr_line'], format='%b-%Y')

# Extract the issue year (loan vintage) - useful because risk varies by economic era
clean_df['issue_year'] = clean_df['issue_d'].dt.year

# Credit history length: how many years between oldest credit line and this loan
clean_df['credit_history_years'] = clean_df['issue_d'].dt.year - clean_df['earliest_cr_line'].dt.year

# Fill the few nulls (from missing earliest_cr_line) with the median
clean_df['credit_history_years'] = clean_df['credit_history_years'].fillna(
    clean_df['credit_history_years'].median()
)

# Drop the raw datetime columns now that we've extracted what we need
clean_df = clean_df.drop(columns=['issue_d', 'earliest_cr_line'])

# Verify
clean_df['credit_history_years'].describe()

count    1.338224e+06
mean     1.626537e+01
std      7.508482e+00
min      3.000000e+00
25%      1.100000e+01
50%      1.500000e+01
75%      2.000000e+01
max      8.300000e+01
Name: credit_history_years, dtype: float64

In [20]:
#Show remainging non numerical vlaues and what they look like. 
non_numeric = [c for c in clean_df.columns if not pd.api.types.is_numeric_dtype(clean_df[c])]
for c in non_numeric:
    print(c, clean_df[c].dtype)

for c in non_numeric:
    print("=== ", c, " ===")
    print(clean_df[c].value_counts(dropna=False).head(10))
    print("")
print("END")

END


In [21]:
clean_df.head()

,loan_amnt,funded_amnt,int_rate,grade,sub_grade,emp_length,annual_inc,pymnt_plan,dti,delinq_2yrs,...,addr_state_VA,addr_state_VT,addr_state_WA,addr_state_WI,addr_state_WV,addr_state_WY,verification_status_Source Verified,verification_status_Verified,issue_year,credit_history_years
100,30000,30000,22.35,3,19,5.0,100000.0,0,30.46,0.0,...,0,0,0,0,0,0,1,0,2018,6
152,40000,40000,16.14,2,13,0.0,45000.0,0,50.53,0.0,...,0,0,0,0,0,0,0,1,2018,9
170,20000,20000,7.56,0,2,10.0,100000.0,0,18.92,0.0,...,0,0,1,0,0,0,0,0,2018,19
186,4500,4500,11.31,1,7,10.0,38500.0,0,4.64,0.0,...,0,0,0,0,0,0,0,0,2018,15
215,8425,8425,27.27,4,24,3.0,450000.0,0,12.37,0.0,...,0,0,0,0,0,0,0,1,2018,21


In [22]:
clean_df.shape

(1338224, 152)

In [23]:
#Check is all rows have a grade.
missing = clean_df[clean_df['grade'].notna() & clean_df['sub_grade'].isna()]
print("Rows with grade but no sub_grade:", len(missing))

missing_rev = clean_df[clean_df['sub_grade'].notna() & clean_df['grade'].isna()]
print("Rows with sub_grade but no grade:", len(missing_rev))

Rows with grade but no sub_grade: 0
Rows with sub_grade but no grade: 0


In [24]:
# Save the cleaned DataFrame to a CSV file
# clean_df.to_csv("~/Optimizing-P2P-Loan-Thresholds-for-Borrowers-Using-LendingClub-Data-/dataset/clean_loans.csv", index=False)

In [25]:
# Correlate every column with the target
# corr = clean_df.corr()['profitable'].drop('profitable').sort_values(key=abs, ascending=False)
# corr

In [26]:
# Undersampling hte majority case
# Want to create a balnced data set for cleaning

df_profit = clean_df[clean_df['profitable'] == 1]
df_loss = clean_df[clean_df['profitable'] == 0]

# create a new DataFrame with the same number of profitable loans as unprofitable loans
df_profit_undersampled = df_profit.sample(n=len(df_loss), random_state=42)

# combine the undersampled profits with the origianl losses
df_undersampled_clean = pd.concat([df_profit_undersampled, df_loss])

# Shuffle the rows of the new DataFrame to mix the profitable and unprofitable loans
df_undersampled_clean = df_undersampled_clean.sample(frac=1, random_state=42).reset_index(drop=True)

df_undersampled_clean.shape

(549174, 152)

In [27]:
# Save the cleaned DataFrame to a CSV file
df_undersampled_clean.to_csv("~/Optimizing-P2P-Loan-Thresholds-for-Borrowers-Using-LendingClub-Data-/dataset/clean_undersampled.csv", index=False)
print("undersampled csv built")

undersampled csv built


NameError: name 'best_xgb' is not defined